# 7. Pipeline: Score a New Quarter

The proposal asked for one reproducible pipeline, not two separate analyses. This notebook is that last step: it
takes the cleaned cases of one FAERS quarter, as if they had just arrived, and

1. builds the same 440 features as notebook 4, using only the artifacts saved from training
   (`data/features/preprocessor.joblib`: scaler, one-hot levels, top-400 drugs, top-10 countries);
2. scores each case with the model from notebook 5 (`models/serious_outcome_model.joblib`): the raw score ranks the
   review queue, the isotonic-calibrated value is the probability shown to a reviewer;
3. flags cases above the validation-chosen threshold for review;
4. tags each case with the safety-profile clusters of its suspect drugs (notebook 6,
   `data/unsupervised/drug_clusters.parquet`).

Nothing is refit here. To score a future quarter, run notebooks 1-2 on its zip and set `QUARTER` below.

**Input:** `data/clean/`, `data/features/preprocessor.joblib`, `models/serious_outcome_model.joblib`,
`data/unsupervised/drug_clusters.parquet`. **Output:** `reports/pipeline_scored_<quarter>.parquet` and a summary.

In [1]:
from __future__ import annotations

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from sklearn.metrics import average_precision_score, roc_auc_score

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

QUARTER = "2026Q2"  # the most recent quarter; part of the test period, never used for fitting

prep = joblib.load(PROJECT_ROOT / "data" / "features" / "preprocessor.joblib")
scorer = joblib.load(PROJECT_ROOT / "models" / "serious_outcome_model.joblib")
drug_clusters = pd.read_parquet(PROJECT_ROOT / "data" / "unsupervised" / "drug_clusters.parquet")
print(f"model: {scorer['family']}, review threshold (calibrated scale): {scorer['calibrated_threshold']:.3f}")

model: gradient boosting, review threshold (calibrated scale): 0.325


## 1. New cases

Legacy cases (first received before 2020, notebook 4) are left out, as in training. Mass reports are kept: in a
live setting nobody knows yet whether a report will turn out to be one of many copies.

In [2]:
reports = pd.read_parquet(CLEAN_DIR / "reports_clean.parquet")
new = reports[(reports["quarter"] == QUARTER) & ~reports["is_legacy_case"]].reset_index(drop=True)
ingredients = pq.read_table(CLEAN_DIR / "case_ingredients", columns=["safetyreportid", "ingredient"]).to_pandas()
ingredients = ingredients[ingredients["safetyreportid"].isin(new["safetyreportid"])]
print(f"{QUARTER}: {len(new):,} cases, {len(ingredients):,} suspect-ingredient rows")

2026Q2: 416,272 cases, 675,117 suspect-ingredient rows


## 2. Features (same recipe as notebook 4, no refitting)

In [3]:
def missing_label(s: pd.Series) -> pd.Series:
    return s.astype("string").fillna("missing")


country = new["occur_country"].astype("string")
base = pd.DataFrame({
    "age_years": new["age_years"].astype("float32"),
    "age_missing": new["age_years"].isna().astype("int8"),
    "age_group": missing_label(new["age_group"]),
    "sex": missing_label(new["sex"]),
    "weight_kg": new["weight_kg"].astype("float32"),
    "weight_missing": new["weight_kg"].isna().astype("int8"),
    "reporter": missing_label(new["reporter"]),
    "report_type": missing_label(new["report_type"]),
    "is_us": country.eq("US").fillna(False).astype("int8"),
    "country": country.where(country.isin(prep["top_countries"]), "OTHER").fillna("missing"),
    "n_drugs": new["n_drugs"].astype("int32"),
    "n_suspect_ingredients": new["n_suspect_ingredients"].astype("int32"),
})

top_drugs = pd.Index(prep["top_drugs"])
rows = pd.Index(new["safetyreportid"]).get_indexer(ingredients["safetyreportid"])
cols = top_drugs.get_indexer(ingredients["ingredient"].astype(str))
keep = cols >= 0
drug_matrix = sp.csr_matrix((np.ones(int(keep.sum()), dtype=np.float32), (rows[keep], cols[keep])),
                            shape=(len(new), len(top_drugs)))
drug_matrix.data[:] = 1
base["n_other_ingredients"] = (base["n_suspect_ingredients"] - np.asarray(drug_matrix.sum(axis=1)).ravel()).astype("int32")

X_new = sp.hstack([prep["preprocessor"].transform(base), drug_matrix], format="csr", dtype=np.float32)
assert X_new.shape[1] == len(scorer["feature_names"])
print("feature matrix:", X_new.shape)

feature matrix: (416272, 440)


## 3. Score and flag

In [4]:
p_raw = np.concatenate([scorer["model"].predict_proba(X_new[i:i + 200_000].toarray())[:, 1]
                        for i in range(0, X_new.shape[0], 200_000)])
new["score"] = p_raw                                     # ranking score (isotonic ties would blur the order)
new["p_serious"] = scorer["calibrator"].predict(p_raw)   # calibrated probability, for display
new["flag_for_review"] = p_raw >= scorer["threshold"]
print(f"flagged for review: {new['flag_for_review'].mean():.1%} of cases")

flagged for review: 38.7% of cases


## 4. Tag with drug safety-profile clusters

A case gets every k-means cluster that one of its suspect drugs belongs to; drugs outside the 500 clustered
ingredients get none.

In [5]:
cluster_of = drug_clusters.set_index("ingredient")["kmeans_cluster"]
label_of = drug_clusters.drop_duplicates("kmeans_cluster").set_index("kmeans_cluster")["kmeans_label"]
tags = ingredients.assign(cluster=ingredients["ingredient"].astype(str).map(cluster_of)).dropna(subset=["cluster"])
tags = tags.groupby("safetyreportid")["cluster"].agg(lambda c: sorted(set(c.astype(int))))
new["drug_clusters"] = new["safetyreportid"].map(tags)
print(f"cases with at least one clustered drug: {new['drug_clusters'].notna().mean():.1%}")

cases with at least one clustered drug: 84.8%


## 5. What a reviewer would see

Because this quarter's outcomes are already known, we can also check the queue.

In [6]:
known = new["y_hosp_or_death"].notna()
y_true = new.loc[known, "y_hosp_or_death"].astype(int)
flag = new.loc[known, "flag_for_review"]
summary = pd.Series({
    "cases scored": len(new),
    "flagged for review": int(new["flag_for_review"].sum()),
    "average precision (known outcome)": average_precision_score(y_true, new.loc[known, "score"]),
    "ROC AUC (known outcome)": roc_auc_score(y_true, new.loc[known, "score"]),
    "precision of the flag": y_true[flag].mean(),
    "recall of the flag": flag[y_true == 1].mean(),
    "mean predicted vs observed rate": f"{new.loc[known, 'p_serious'].mean():.3f} vs {y_true.mean():.3f}",
})
display(summary.to_frame(QUARTER))

exploded = new.explode("drug_clusters").dropna(subset=["drug_clusters"])
by_cluster = (exploded.groupby("drug_clusters")
              .agg(cases=("safetyreportid", "size"), flagged_share=("flag_for_review", "mean"),
                   mean_p_serious=("p_serious", "mean"))
              .assign(label=lambda t: label_of.reindex(t.index.astype(int)).to_numpy())
              .sort_values("mean_p_serious", ascending=False))
print("review queue by drug cluster:")
display(by_cluster.round(3).head(12))

queue = new.sort_values("score", ascending=False)[
    ["safetyreportid", "p_serious", "flag_for_review", "drug_clusters", "reporter", "occur_country", "age_years", "sex"]]
display(queue.head(10))

out = PROJECT_ROOT / "reports" / f"pipeline_scored_{QUARTER}.parquet"
new[["safetyreportid", "quarter", "score", "p_serious", "flag_for_review", "drug_clusters"]].assign(
    drug_clusters=lambda t: t["drug_clusters"].map(lambda c: ",".join(map(str, c)) if isinstance(c, list) else "")
).to_parquet(out, index=False)
print("saved", out)

,2026Q2
cases scored,416272
flagged for review,160974
average precision (known outcome),0.580304
ROC AUC (known outcome),0.817291
precision of the flag,0.476771
recall of the flag,0.74806
mean predicted vs observed rate,0.266 vs 0.246


review queue by drug cluster:


,cases,flagged_share,mean_p_serious,label
drug_clusters,,,,
3,3331,0.835,0.547,DRUG ABUSE / DRUG USE DISORDER / POISONING DELIBERATE
5,2668,0.794,0.543,NAUSEA / VOMITING / DRUG INEFFECTIVE
26,3494,0.890,0.524,OFF LABEL USE / DRUG INEFFECTIVE / PATHOGEN RESISTANCE
8,22890,0.877,0.501,FEBRILE NEUTROPENIA / MYELOSUPPRESSION / OFF LABEL USE
23,8842,0.731,0.494,DRUG WITHDRAWAL SYNDROME NEONATAL / DEPENDENCE / LEARNING DISABILITY
4,4008,0.609,0.461,ANTI-CYCLIC CITRULLINATED PEPTIDE ANTIBODY POSITIVE / PEMPHIGUS / ...
27,29400,0.726,0.454,OFF LABEL USE / FATIGUE / DRUG INEFFECTIVE
22,7041,0.630,0.452,ON AND OFF PHENOMENON / PARKINSON^S DISEASE / FALL
13,18060,0.680,0.448,ACUTE KIDNEY INJURY / DRUG INEFFECTIVE / DIZZINESS


,safetyreportid,p_serious,flag_for_review,drug_clusters,reporter,occur_country,age_years,sex
11675,24675380,1.0,True,"[2, 4, 13, 16, 18, 19, 20, 23]",other_hcp,CA,44.0,female
386633,26915464,1.0,True,"[2, 4, 13, 16, 18, 19, 23, 29]",other_hcp,CA,44.0,female
356074,26881105,1.0,True,"[2, 4, 13, 16, 18, 19, 23, 29]",other_hcp,CA,44.0,female
398051,26928295,1.0,True,"[2, 4, 13, 16, 18, 19, 23, 29]",other_hcp,CA,44.0,female
365288,26891441,1.0,True,"[2, 4, 13, 16, 18, 19, 23, 29]",other_hcp,CA,44.0,female
38354,26250584,1.0,True,"[2, 4, 13, 16, 18, 19, 23, 29]",other_hcp,CA,44.0,female
4383,22758940,1.0,True,"[2, 4, 13, 16, 18, 19, 20, 23, 29]",other_hcp,CA,44.0,female
23908,25724887,1.0,True,"[2, 4, 9, 13, 16, 18, 19, 20, 23]",physician,CA,44.0,female
345411,26869287,1.0,True,"[2, 4, 13, 16, 18, 19, 23, 29]",other_hcp,CA,44.0,female
12111,24748345,1.0,True,"[2, 4, 13, 16, 18, 19, 20, 23, 29]",other_hcp,CA,44.0,female


saved /Users/suvidhinikhil/Data science projects/Milestone 2 projects/FAERS_Milestone2_Final/reports/pipeline_scored_2026Q2.parquet
